In [25]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix

In [26]:
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'], dtype={'gameId' : str, 'H_teamId' : str, 'A_teamId' : str,})
data = data.round(2)

In [27]:
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
#data_test = data_test.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])
data_train = data[~condition]
#data_train = data_train.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

In [28]:
X_train = data_train.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId'])  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId'])  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
X_test

array([[0.41129139, 0.49425287, 0.47058824, ..., 0.76647564, 0.625     ,
        0.49295775],
       [0.4432422 , 0.54597701, 0.52941176, ..., 0.48949379, 0.66666667,
        0.57676056],
       [0.60875817, 0.77586207, 0.70588235, ..., 0.15520535, 0.625     ,
        0.49507042],
       ...,
       [0.51946317, 0.66091954, 0.58823529, ..., 0.1599809 , 0.54166667,
        0.59366197],
       [0.5942564 , 0.62068966, 0.70588235, ..., 0.37249284, 0.5       ,
        0.53943662],
       [0.49087775, 0.45402299, 0.58823529, ..., 0.53963706, 0.375     ,
        0.56619718]])

In [29]:
# Définir les métriques de performance à calculer
scoring = {'accuracy': make_scorer(accuracy_score), 'f1': make_scorer(f1_score)}

In [30]:
knn = KNeighborsClassifier(n_neighbors=20)

In [31]:
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=20)

In [32]:
y_pred = knn.predict(X_test) 

In [33]:
accuracy = accuracy_score(y_test, y_pred)  # y_test sont les étiquettes de classe réelles des données de test
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("Matrice de confusion :")
print(conf_matrix)

Accuracy: 0.6597285067873303
F1-score: 0.6952998379254457
Matrice de confusion :
[[300 203]
 [173 429]]


In [34]:
data_knn_predict = data_test[['gameId', 'GAME_DATE', 'HOME_WON', 'H_teamId', 'A_teamId']].copy()
data_knn_predict.loc[:, 'PRED'] = y_pred

In [35]:
data_knn_predict

,gameId,GAME_DATE,HOME_WON,H_teamId,A_teamId,PRED
10794,0022300001,2023-11-03,1,1610612754,1610612739,0
10795,0022300002,2023-11-03,1,1610612749,1610612752,0
10796,0022300003,2023-11-03,1,1610612748,1610612764,1
10797,0022300004,2023-11-03,0,1610612741,1610612751,0
10798,0022300005,2023-11-03,0,1610612760,1610612744,1
...,...,...,...,...,...,...
11894,0022301226,2023-12-08,0,1610612757,1610612742,0
11895,0022301227,2023-12-08,1,1610612738,1610612752,1
11896,0022301228,2023-12-08,0,1610612756,1610612758,0
11897,0022301229,2023-12-07,0,1610612749,1610612754,1


In [36]:
dict_team = {
    '1610612747' : "Lakers",
    '1610612744' : "Warriors",
    '1610612738' : "Celtics",
    '1610612739' : "Cavaliers",
    '1610612749' : "Bucks",
    '1610612752' : "Knicks",
    '1610612753' : "Magic",
    '1610612748' : "Heat",
    '1610612755' : "76ers",
    '1610612754' : "Pacers",
    '1610612741' : "Bulls",
    '1610612737' : "Hawks",
    '1610612751' : "Nets",
    '1610612761' : "Raptors",
    '1610612766' : "Hornets",
    '1610612765' : "Pistons",
    '1610612764' : "Wizards",
    '1610612760' : "Thunder",
    '1610612750' : "Wolves",
    '1610612743' : "Nuggets",
    '1610612746' : "Clippers",
    '1610612740' : "Pelicans",
    '1610612756' : "Suns",
    '1610612758' : "kings",
    '1610612742' : "Mavs",
    '1610612745' : "Rockets",
    '1610612762' : "Jazz",
    '1610612763' : "Grizzlies",
    '1610612757' : "Blazers",
    '1610612759' : "Spurs",
}
for team_id, team_name in dict_team.items():
    data_knn_predict.replace(team_id, team_name, inplace=True)


data_knn_predict.set_index('gameId', inplace=True)
data_knn_predict.head(3).to_json("../dataset/KNN_predict.json")